In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec
import geopandas as gpd

import torch
import massbalancemachine as mbm

import math
import string
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import root_mean_squared_error

from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.plotting.palettes import get_color_maps
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.geo_data import apply_gaussian_filter

# reuse _load_lstm_ds from the existing plotting module — reads {rgi_id}/{rgi_id}_{year}_{period}.zarr
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.plotting.spatial_maps import _load_lstm_ds

sys.path.append(os.path.join(os.getcwd(), '../../'))
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

BARPLOT_ALPHA = 0.7

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
print(num_measurements)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Subregions:

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4'
    #'CENTRALASIA', 'ALA', #'CA_12',
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Pooled model assets: {len(assets_full['ds_train'])} sequences")

# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint, scaler_geo = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint, blur_geo = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

### Hold-out set:

In [ ]:
RECOMPUTE_SPLITS = False

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

gl_level_split = []
id_level_split = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

In [ ]:
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

N_REPEATS = 5

# ── Build FT assets for all target subregions, repeated N_REPEATS times ──────
ft_assets_glacier = {}
ft_assets_id = {region: [] for region in TARGET_REGIONS_SUB}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split (unchanged — deterministic by glacier, no repeats needed)
    if region in gl_level_split:
        if region in gl_level_split:
            pool_glaciers = splits[region]["pool_glaciers"]
            holdout_glaciers = splits[region]["holdout_glaciers"]
            exp_key = f"FD_to_{region}"

            ds_ft_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_FT",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    pool_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_TEST",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(
                    holdout_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    holdout_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
            ft_assets_glacier[region] = {
                "ds_ft": ds_ft_gl,
                "ds_test": ds_test_gl,
                "ft_train_idx": ft_train_idx_gl,
                "ft_val_idx": ft_val_idx_gl,
            }
            print(
                f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
            )

    # 2. ID-level split, repeated N_REPEATS times with different seeds
    if region in id_level_split:
        winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "winter"]["ID"].unique().tolist()
        annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "annual"]["ID"].unique().tolist()

        for rep in range(N_REPEATS):
            winter_ids = list(winter_ids_full)
            annual_ids = list(annual_ids_full)

            rep_seed = cfg.seed + rep
            rng = np.random.default_rng(rep_seed)
            rng.shuffle(winter_ids)
            rng.shuffle(annual_ids)

            n_ft_w = max(1, int(len(winter_ids) *
                                FT_FRAC)) if winter_ids else 0
            n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
            ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
            test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

            # keep track of ft_ids per rep for the cross-rep check below
            if rep == 0:
                _ft_id_sets_for_check = []
            _ft_id_sets_for_check.append(set(ft_ids))

            exp_key_id = f"FD_to_{region}_IDsplit_rep{rep}"
            ds_ft_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_TEST",
                df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_id), val_ratio=0.2, seed=rep_seed)

            ft_assets_id[region].append({
                "rep": rep,
                "seed": rep_seed,
                "ds_ft": ds_ft_id,
                "ds_test": ds_test_id,
                "ft_train_idx": ft_train_idx_id,
                "ft_val_idx": ft_val_idx_id,
            })
            print(f"  [ID-split rep {rep}] ds_ft  : {len(ds_ft_id)} seqs | "
                  f"ds_test: {len(ds_test_id)} seqs")

            # ── sanity check after all reps for this region are built ────────────
        if region in id_level_split:
            n_reps = len(_ft_id_sets_for_check)
            identical_pairs = [
                (i, j) for i in range(n_reps) for j in range(i + 1, n_reps)
                if _ft_id_sets_for_check[i] == _ft_id_sets_for_check[j]
            ]
            if identical_pairs:
                print(
                    f"  WARNING [{region}]: identical FT id-sets for repeats {identical_pairs}"
                )
            else:
                overlaps = [
                    len(_ft_id_sets_for_check[i] & _ft_id_sets_for_check[j]) /
                    len(_ft_id_sets_for_check[i]) for i in range(n_reps)
                    for j in range(i + 1, n_reps)
                ]
                print(
                    f"  [check] {region}: {n_reps} repeats distinct (mean overlap {np.mean(overlaps):.1%})"
                )

# convenience dict: glacier-split where available, ID-split (list of reps) for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

## Model assets:

## Transformer:

### Zero shot:

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_pooled, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

print(f"\n{'='*60}")
print("Zero-shot evaluation of pooled model on target regions")
print(f"{'='*60}")

RUN_ZS_EVAL = False  # set True to recompute zero-shot stake metrics

if RUN_ZS_EVAL:
    print(f"\n--- ID-split (mean ± std over {N_REPEATS} repeats) ---")
    zs_id_metrics = {}

    for region in id_level_split:
        rep_metrics = []
        for rep_data in ft_assets_id[region]:
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                rep_data["ds_test"])
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
            metrics, _ = model_pooled.evaluate_with_preds(
                device, test_dl, ds_test_copy)
            rep_metrics.append(metrics)

        zs_id_metrics[region] = rep_metrics
        rmse_a = [m["RMSE_annual"] for m in rep_metrics]
        rmse_w = [m["RMSE_winter"] for m in rep_metrics]
        r2_a = [m["R2_annual"] for m in rep_metrics]
        r2_w = [m["R2_winter"] for m in rep_metrics]
        print(f"  {region:15s}  "
              f"RMSE_a={np.mean(rmse_a):.3f}±{np.std(rmse_a):.3f}  "
              f"RMSE_w={np.mean(rmse_w):.3f}±{np.std(rmse_w):.3f}  "
              f"R2_a={np.mean(r2_a):.3f}±{np.std(r2_a):.3f}  "
              f"R2_w={np.mean(r2_w):.3f}±{np.std(r2_w):.3f}")

### Fine tuned model:

In [ ]:
# ── Train or load fine-tuned models for all non-CEU regions ──────────────────
RGI_TO_FT_REGION = {
    "06": "ISL",
    "07": "SJM",
    "08": "NOR",
    "01": "ALA",
    "11": "CH",
    "13_CA_3": "CA_3",
    "13_CA_4": "CA_4"
}

STRATEGY_OTHER = "full"
ft_models_by_rgi = {}  # no longer pre-populated with CEU
models_dir_ft = BASE_DIR / "models" / "finetuned"

for rgi_id, ft_code in RGI_TO_FT_REGION.items():
    print(f"\n{'='*60}\nRGI {rgi_id} ({ft_code})")
    ft_models_by_rgi[rgi_id] = train_or_load_ft_models_for_region(
        region_code=ft_code,
        monthly_cache=monthly_cache,
        model_pooled=model_pooled,
        models_dir_ft=models_dir_ft,
        strategy=STRATEGY_OTHER,
        n_repeats=N_REPEATS,
        model_date=MODEL_DATE,
        cfg=cfg,
        device=device,
        best_params=PARAMS_TRANSFORMER,
        ds_pooled_scaler=ds_pooled_scaler,
        ft_frac=FT_FRAC,
        force_retrain=False,
        CACHE_DIR=CACHE_DIR)
    print(f"  -> {len(ft_models_by_rgi[rgi_id])} models ready")


### Single source models:

In [ ]:
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION/models"
TRAIN_FLAG = False

single_source_models = {}
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    df_src = res_xreg_by_source[src]["df_train"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    # build per-source scaler
    scaler_src = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler_src.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler_src

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          PARAMS_TRANSFORMER,
                                                          device,
                                                          verbose=False)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=PARAMS_TRANSFORMER,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix="transformer_xreg",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=N_EPOCHS,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

single_source_models_lstm = {}

for src in SOURCE_REGIONS:
    ckpt_lstm = XREG_MODELS_BASE / src / f"lstm_xreg_{src}_{MODEL_DATE}.pt"

    if not ckpt_lstm.exists():
        print(f"  No LSTM checkpoint found for {src}: {ckpt_lstm}, skipping")
        continue

    m_lstm = mbm.models.LSTM_MB.build_model_from_params(cfg,
                                                        PARAMS_LSTM,
                                                        device,
                                                        verbose=False)
    state = torch.load(ckpt_lstm, map_location=device)
    m_lstm.load_state_dict(state)
    m_lstm.eval()
    single_source_models_lstm[src] = m_lstm
    print(f"  Loaded LSTM {src} from {ckpt_lstm}")

print(
    f"Loaded {len(single_source_models_lstm)} LSTM single-source models: {list(single_source_models_lstm)}"
)

## Make predictions:

In [ ]:
# import shutil

# paths_to_clear = [
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_11_ZS"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_08_ZS"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_11_SR_CH"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_08_SR_NOR"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_11_SR_CH_LSTM"),
#     os.path.join(cfg.dataPath, "predictions_transformer", "RGI_08_SR_NOR_LSTM"),
#     *[os.path.join(cfg.dataPath, "predictions_transformer", f"RGI_11_CEU_rep{r}") for r in range(N_REPEATS)],
#     *[os.path.join(cfg.dataPath, "predictions_transformer", f"RGI_11_CH_rep{r}") for r in range(N_REPEATS)],
#     *[os.path.join(cfg.dataPath, "predictions_transformer", f"RGI_08_NOR_rep{r}") for r in range(N_REPEATS)],
# ]

# for p in paths_to_clear:
#     if os.path.exists(p):
#         shutil.rmtree(p)
#         print(f"Cleared: {p}")

### Specific glaciers:

In [ ]:
# GLACIER_CONFIG = {
#     "aletsch": {
#         "rgi_ids":
#         monthly_cache["CH"]["data_monthly"].query("GLACIER == 'aletsch'")
#         ["RGIId"].unique().tolist(),
#         "region_folder":
#         "RGI_11_CentralEurope",
#         "rgi_id_str":
#         "11",
#         "ft_local_key":
#         "11_CH",
#         "sr_tf_key":
#         "CH",
#         "sr_lstm_key":
#         "CH",
#         "df_stakes":
#         dfs["11"],
#     },
#     "rhone": {
#         "rgi_ids":
#         monthly_cache["CH"]["data_monthly"].query("GLACIER == 'rhone'")
#         ["RGIId"].unique().tolist(),
#         "region_folder":
#         "RGI_11_CentralEurope",
#         "rgi_id_str":
#         "11",
#         "ft_local_key":
#         "11_CH",
#         "sr_tf_key":
#         "CH",
#         "sr_lstm_key":
#         "CH",
#         "df_stakes":
#         dfs["11"],
#     },
#     "Nigardsbreen": {
#         "rgi_ids":
#         monthly_cache["NOR"]["data_monthly"].query("GLACIER == 'Nigardsbreen'")
#         ["RGIId"].unique().tolist(),
#         "region_folder":
#         "RGI_08_Scandinavia",
#         "rgi_id_str":
#         "08",
#         "ft_local_key":
#         "08_NOR",
#         "sr_tf_key":
#         "NOR",
#         "sr_lstm_key":
#         "NOR",
#         "df_stakes":
#         dfs["08"],
#     },
# }

### All stakes:

In [ ]:
rgi_regions = {
    "01": {
        "name": "Alaska",
        "folder": "RGI_01_Alaska",
        "file": "01_rgi60_Alaska.shp",
        "code": "ALA",
        "era5_source": "EU_US_CANADA"
    },
    "06": {
        "name": "Iceland",
        "folder": "RGI_06_Iceland",
        "file": "06_rgi60_Iceland.shp",
        "code": "ISL",
        "era5_source": "EU_US_CANADA"
    },
    "07": {
        "name": "Svalbard",
        "folder": "RGI_07_Svalbard",
        "file": "07_rgi60_Svalbard.shp",
        "code": "SJM",
        "era5_source": "EU_US_CANADA"
    },
    "08": {
        "name": "Scandinavia",
        "folder": "RGI_08_Scandinavia",
        "file": "08_rgi60_Scandinavia.shp",
        "code": "SCA",
        "subregions": ["Norway"],
        "subregions_codes": ["NOR"],
        "era5_source": "EU_US_CANADA"
    },
    "11": {
        "name": "CentralEurope",
        "folder": "RGI_11_CentralEurope",
        "file": "11_rgi60_CentralEurope.shp",
        "code": "CEU",
        "subregions": ["France", "Switzerland", "Italy_Austria"],
        "subregions_codes": ["FR", "CH", "IT", "AT"],
        "era5_source": "EU_US_CANADA"
    },
    "13": {
        "name": "CentralAsia",
        "folder": "RGI_13_CentralAsia",
        "file": "13_rgi60_CentralAsia.shp",
        "code": "CENTRALASIA",
        "subregions_codes": ["CA_3", "CA_4"],
        "era5_source": "HMA"
    },
}

ft_models_for_pred = {
    "11_CH": ft_models_by_rgi["11"],
    "08_NOR": ft_models_by_rgi["08"],
    "01_ALA": ft_models_by_rgi["01"],
    "13_CA_3": ft_models_by_rgi["13_CA_3"],
    "13_CA_4": ft_models_by_rgi["13_CA_4"],
    "07_SJM": ft_models_by_rgi["07"],
    "11_FR": ft_models_by_rgi["11"],
    "11_IT": ft_models_by_rgi["11"],
    "11_AT": ft_models_by_rgi["11"]

}

WANTED_REGION_CODES = ["CH", "FR", "ISL", "ALA", "NOR", "CA_3", "CA_4", "SJM", "IT", "AT"]


def resolve_rgi_region(region_code, rgi_regions=rgi_regions):
    """
    Map a monthly_cache region code (e.g. 'CH', 'NOR', 'CENTRALASIA') to its
    (rgi_id_str, region_info) entry in rgi_regions, checking both top-level
    'code' and 'subregions_codes'.
    """
    for rgi_id_str, info in rgi_regions.items():
        if info["code"] == region_code:
            return rgi_id_str, info
        if region_code in info.get("subregions_codes", []):
            return rgi_id_str, info
    raise KeyError(f"{region_code!r} not found in rgi_regions")


# dynamically extend ft_models_for_pred beyond the 3 keys hardcoded in cell 24,
# so any region trained via RGI_TO_FT_REGION is picked up automatically
ft_models_for_pred_all = dict(ft_models_for_pred)  # keep your existing entries
for rgi_id_str, ft_code in RGI_TO_FT_REGION.items():
    key = f"{rgi_id_str}_{ft_code}"
    if key not in ft_models_for_pred_all and rgi_id_str in ft_models_by_rgi:
        ft_models_for_pred_all[key] = ft_models_by_rgi[rgi_id_str]


def make_glacier_config_for_region(region_code, monthly_cache, dfs):
    if region_code not in monthly_cache:
        raise KeyError(f"{region_code!r} not in monthly_cache")

    rgi_id_str, info = resolve_rgi_region(region_code)
    region_folder = info["folder"]
    ft_local_key = f"{rgi_id_str}_{region_code}"
    sr_key = region_code if region_code in SOURCE_REGIONS else None
    df_stakes = dfs.get(rgi_id_str)

    glaciers = monthly_cache[region_code]["data_monthly"]["GLACIER"].unique()

    return {
        glacier: {
            "rgi_ids":
            monthly_cache[region_code]["data_monthly"].query(
                f"GLACIER == '{glacier}'")["RGIId"].unique().tolist(),
            "region_folder":
            region_folder,
            "rgi_id_str":
            rgi_id_str,
            "ft_local_key":
            ft_local_key if ft_local_key in ft_models_for_pred_all else None,
            "sr_tf_key":
            sr_key if sr_key in single_source_models else None,
            "sr_lstm_key":
            sr_key if sr_key in single_source_models_lstm else None,
            "df_stakes":
            df_stakes,
        }
        for glacier in glaciers
    }


ALL_REGION_CONFIGS = {}
for region_code in WANTED_REGION_CODES:
    try:
        cfg_dict = make_glacier_config_for_region(region_code, monthly_cache,
                                                  dfs)
    except KeyError as e:
        print(f"  [skip] {region_code}: {e}")
        continue
    ALL_REGION_CONFIGS[region_code] = cfg_dict
    globals()[f"GLACIER_CONFIG_{region_code}"] = cfg_dict
    n_ft = sum(1 for g in cfg_dict.values() if g["ft_local_key"] is not None)
    print(f"{len(cfg_dict)} {region_code} stake glaciers "
          f"({n_ft} with FT models available): {list(cfg_dict.keys())}")

ALL_GLACIER_CONFIG = {
    name: gcfg
    for cfg_dict in ALL_REGION_CONFIGS.values()
    for name, gcfg in cfg_dict.items()
}

### Predictions:

In [ ]:
from tqdm.auto import tqdm

PRED_BASE = os.path.join(cfg.dataPath,
                         "cache/pooled_pred/predictions_transformer")


def run_region(region_code, glacier_names=None, empty_cache_every=1):
    """
    Run ZS + FT prediction generation for a single region_code only.
    Call this one region at a time — in separate cells, and restart the
    kernel in between if you want a fully clean GPU/CPU state — rather
    than looping through all regions back-to-back in one run.

    Parameters
    ----------
    region_code : str
        Key into ALL_REGION_CONFIGS, e.g. "ISL", "CH", "CA_3".
    glacier_names : list[str], optional
        If given, only run these glaciers within the region instead of
        all of them, e.g. glacier_names=["Nigardsbreen"].
    """
    if region_code not in ALL_REGION_CONFIGS:
        raise KeyError(
            f"{region_code!r} not in ALL_REGION_CONFIGS. Available: {list(ALL_REGION_CONFIGS)}"
        )

    cfg_dict = ALL_REGION_CONFIGS[region_code]

    if glacier_names is not None:
        missing = [g for g in glacier_names if g not in cfg_dict]
        if missing:
            raise KeyError(f"{missing} not found in region {region_code!r}. "
                           f"Available: {list(cfg_dict)}")
        cfg_dict = {g: cfg_dict[g] for g in glacier_names}

    pbar = tqdm(cfg_dict.items(), total=len(cfg_dict), desc=region_code)

    for glacier_name, gcfg in pbar:
        pbar.set_description(f"{region_code}:{glacier_name}")
        folder = gcfg["region_folder"]
        rgi_id_str = gcfg["rgi_id_str"]
        glacier_filter = gcfg["rgi_ids"]
        ds_cache_dir = os.path.join(cfg.dataPath, "RGI_v6", folder,
                                    "grid_ds_cache")

        # --- Multi-region Zero-Shot ---
        predict_grid_from_cached_datasets(
            cfg=cfg,
            ds_cache_dir=ds_cache_dir,
            model=model_pooled,
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            region_folder=folder,
            path_save_glw=os.path.join(PRED_BASE, f"RGI_{rgi_id_str}_ZS"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            glacier_filter=glacier_filter,
            period="annual",
            empty_cache_every=empty_cache_every,
        )

        # --- Multi-region Fine-Tuned ---
        ft_local_key = gcfg["ft_local_key"]
        if ft_local_key and ft_local_key in ft_models_for_pred_all:
            for rep, ft_model in ft_models_for_pred_all[ft_local_key]:
                if rep == 0:
                    predict_grid_from_cached_datasets(
                        cfg=cfg,
                        ds_cache_dir=ds_cache_dir,
                        model=ft_model,
                        ds_pooled_scaler=ds_pooled_scaler,
                        device=device,
                        region_folder=folder,
                        path_save_glw=os.path.join(
                            PRED_BASE,
                            f"RGI_{rgi_id_str}_FT_{ft_local_key}_rep{rep}"),
                        months_head_pad=months_head_pad,
                        months_tail_pad=months_tail_pad,
                        glacier_filter=glacier_filter,
                        period="annual",
                        empty_cache_every=empty_cache_every,
                    )
                else:
                    continue
        else:
            pbar.write(
                f"  [no FT] {glacier_name}: no fine-tuned model for {ft_local_key}"
            )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print(f"Done with region {region_code} ({len(cfg_dict)} glaciers).")

In [ ]:
# run_region("CH")

In [ ]:
# run_region("SJM")

In [ ]:
# run_region("FR")

In [ ]:
# run_region("ISL")

In [ ]:
# run_region("ALA")

In [ ]:
# run_region("NOR")

In [ ]:
# run_region("CA_3")

In [ ]:
# run_region("CA_4")

In [ ]:
run_region("IT")
run_region("AT")

## Plots:

### Stake locations:

In [ ]:
# fig = plot_pred_vs_obs_stakes(
#     glaciers={
#         name: (gcfg["rgi_ids"][0], gcfg["df_stakes"])
#         for name, gcfg in GLACIER_CONFIG.items()
#     },
#     models={
#         "Zero-Shot (LSTM)": {
#             name:
#             os.path.join(
#                 PRED_BASE,
#                 f"RGI_{gcfg['rgi_id_str']}_SR_{gcfg['sr_lstm_key']}_LSTM")
#             if gcfg["sr_lstm_key"] in single_source_models_lstm else None
#             for name, gcfg in GLACIER_CONFIG.items()
#         },
#         "Zero-Shot (Transformer)": {
#             name:
#             os.path.join(PRED_BASE,
#                          f"RGI_{gcfg['rgi_id_str']}_SR_{gcfg['sr_tf_key']}")
#             if gcfg["sr_tf_key"] in single_source_models else None
#             for name, gcfg in GLACIER_CONFIG.items()
#         },
#     },
#     years=list(range(2005, 2025)),
#     period="annual",
# )

In [ ]:
# # scatter
# fig = plot_pred_vs_obs_stakes(
#     glaciers={
#         name: (gcfg["rgi_ids"][0], gcfg["df_stakes"])
#         for name, gcfg in GLACIER_CONFIG_ALA.items()
#     },
#     models={
#         "Multi-region ZS": {
#             name: os.path.join(PRED_BASE, "RGI_01_ZS")
#             for name in GLACIER_CONFIG_ALA
#         },
#         "FT local (ALA)": {
#             name: [
#                 os.path.join(PRED_BASE, f"RGI_01_FT_01_ALA_rep{rep}")
#                 for rep, _ in ft_models_for_pred["01_ALA"]
#             ]
#             for name in GLACIER_CONFIG_ALA
#         },
#     },
#     years=list(range(2005, 2025)),
#     period="annual",
# )

In [ ]:
# figs = plot_pred_vs_obs_stakes_grid(
#     glaciers={
#         name: (gcfg["rgi_ids"][0], gcfg["df_stakes"])
#         for name, gcfg in GLACIER_CONFIG_ALA.items()
#     },
#     models={
#         "Multi-region ZS": {
#             name: os.path.join(PRED_BASE, "RGI_01_ZS")
#             for name in GLACIER_CONFIG_ALA
#         },
#     },
#     years=list(range(2005, 2025)),
#     period="annual",
#     ncols=4,
# )

### Gradients:

In [ ]:
# fig = plot_transformer_gradients(
#     GLACIER_CONFIG=GLACIER_CONFIG,
#     PRED_BASE=PRED_BASE,
#     models={
#         "SR ZS (Transformer)": "RGI_{rgi_id_str}_SR_{sr_tf_key}",
#         "SR ZS (LSTM)": "RGI_{rgi_id_str}_SR_{sr_lstm_key}_LSTM",
#     },
#     period="annual",
# )

In [ ]:
# fig = plot_transformer_gradients(
#     GLACIER_CONFIG=GLACIER_CONFIG_ALA,
#     PRED_BASE=PRED_BASE,
#     models={
#         "Multi-region ZS": "RGI_01_ZS",  # fixed path, no template needed,
#         "Multi-region FT":
#         "RGI_01_FT_01_ALA_rep0",  # fixed path, no template needed
#     },
#     period="annual",
#     ncols=3)

### Maps:

In [ ]:
# for glacier_name, gcfg in GLACIER_CONFIG.items():
#     fig, fig_cb = plot_transformer_grids(
#         glacier_name=glacier_name,
#         rgi_id=gcfg["rgi_ids"][0],
#         years=list(range(2005, 2025)),
#         path_pred_transformer=os.path.join(
#             PRED_BASE, f"RGI_{gcfg['rgi_id_str']}_SR_{gcfg['sr_tf_key']}"),
#         df_stakes=gcfg["df_stakes"],
#         period="annual",
#     )
#     plt.show()

In [ ]:
# glacier_name = 'GULKANA'
# rgn_code, rgn_number = 'ALA', '01'
# gcfg = ALL_REGION_CONFIGS[rgn_code][glacier_name]
# fig, fig_cb = plot_transformer_grids(
#     glacier_name=glacier_name,
#     rgi_id=gcfg["rgi_ids"][0],
#     years=list(range(2005, 2025)),
#     path_pred_transformer=os.path.join(PRED_BASE, f"RGI_{rgn_number}_ZS"),
#     df_stakes=gcfg["df_stakes"],
#     period="annual",
# )
# plt.show()

In [ ]:
# glacier_name = 'Nigardsbreen'
# rgn_code, rgn_number = 'NOR', '08'
# gcfg = ALL_REGION_CONFIGS[rgn_code][glacier_name]
# fig, fig_cb = plot_transformer_grids(
#     glacier_name=glacier_name,
#     rgi_id=gcfg["rgi_ids"][0],
#     years=list(range(2005, 2025)),
#     path_pred_transformer=os.path.join(PRED_BASE, f"RGI_{rgn_number}_ZS"),
#     df_stakes=gcfg["df_stakes"],
#     period="annual",
# )
# plt.show()

In [ ]:
# glacier_name = 'Thjorsarjoekull (Hofsjoekull E)'
# rgn_code, rgn_numb = 'ISL', '06'
# gcfg = ALL_REGION_CONFIGS[rgn_code][glacier_name]
# fig, fig_cb = plot_transformer_grids(
#     glacier_name=glacier_name,
#     rgi_id=gcfg["rgi_ids"][0],
#     years=list(range(2005, 2025)),
#     path_pred_transformer=os.path.join(PRED_BASE, f"RGI_{rgn_numb}_ZS"),
#     df_stakes=gcfg["df_stakes"],
#     period="annual",
# )
# plt.show()

In [ ]:
# ICELAND_GLACIERS_OF_INTEREST = [
#     "Bruarjoekull",
#     "Tungnaarjoekull",
#     "Koeldukvislarjoekull",
#     "Sidujoekull/Skaftarjoekull",
#     "Dyngjujoekull",
#     "Mulajoekull",
#     "Eyjabakkajoekull",
#     "Hagafellsjoekull East (Langjoekull S Dome)",
#     "Hagafellsjoekull West",
#     "Thjorsarjoekull (Hofsjoekull E)",
#     "Blagnipujoekull (Hofsjoekull SW)",
#     "Breidamerkurjoekull",
#     "Skeidararjoekull",
#     "Langjoekull Ice Cap",
# ]
# rgn_code, rgn_numb = 'ISL', '06'

# for glacier_name in ICELAND_GLACIERS_OF_INTEREST:
#     gcfg = ALL_REGION_CONFIGS[rgn_code][glacier_name]
#     fig, fig_cb = plot_transformer_grids(
#         glacier_name=glacier_name,
#         rgi_id=gcfg["rgi_ids"][0],
#         years=list(range(2005, 2025)),
#         path_pred_transformer=os.path.join(PRED_BASE, f"RGI_{rgn_numb}_ZS"),
#         df_stakes=gcfg["df_stakes"],
#         period="annual",
#     )
#     plt.close(fig_cb)  # discard the colorbar figure before anything gets displayed
#     plt.show()          # display just the main grid figure

In [ ]:
# rgn_code, rgn_numb = 'CA_3', '13'
# for glacier_name, gcfg in ALL_REGION_CONFIGS['CA_3'].items():
#     gcfg = ALL_REGION_CONFIGS[rgn_code][glacier_name]
#     fig, fig_cb = plot_transformer_grids(
#         glacier_name=glacier_name,
#         rgi_id=gcfg["rgi_ids"][0],
#         years=list(range(2005, 2025)),
#         path_pred_transformer=os.path.join(PRED_BASE, f"RGI_{rgn_numb}_ZS"),
#         df_stakes=gcfg["df_stakes"],
#         period="annual",
#     )
#     plt.close(
#         fig_cb)  # discard the colorbar figure before anything gets displayed
#     plt.show()  # display just the main grid figure

In [ ]:
# rgn_code, rgn_numb = 'ALA', '01'
# for glacier_name, gcfg in ALL_REGION_CONFIGS[rgn_code].items():
#     gcfg = ALL_REGION_CONFIGS[rgn_code][glacier_name]
#     fig, fig_cb = plot_transformer_grids(
#         glacier_name=glacier_name,
#         rgi_id=gcfg["rgi_ids"][0],
#         years=list(range(2005, 2025)),
#         path_pred_transformer=os.path.join(PRED_BASE, f"RGI_{rgn_numb}_ZS"),
#         df_stakes=gcfg["df_stakes"],
#         period="annual",
#     )
#     plt.close(
#         fig_cb)  # discard the colorbar figure before anything gets displayed
#     plt.show()  # display just the main grid figure

In [ ]:
# ALL_REGION_CONFIGS['CA_3'].keys()

In [ ]:
# ALL_REGION_CONFIGS['ALA'].keys()

#### Specific glaciers:

In [ ]:
def plot_glacier_grid(
    glacier_years,  # list of {glacier_name, rgi_id, years, path_pred, df_stakes}
    period="annual",
    apply_smoothing=True,
    ncols=4,
    text_size=7,
    figsize_col=8,  # cm per column
    figsize_row=10,  # cm per row
    cb_width_ratio=0.06,  # width of colorbar column relative to a map column
    cm=1 / 2.54,
):
    """
    Grid of glacier MB maps, one row per glacier, ncols years per row.
    Each row gets its own color scale and its own colorbar (glaciers
    can have very different MB ranges, so a single global scale washes
    some of them out).
    """
    # ── load datasets and build one entry per row (glacier), with its own vmin/vmax ──
    row_data = []
    for entry in glacier_years:
        glacier_name = entry["glacier_name"]
        rgi_id = entry["rgi_id"]
        path_pred = entry["path_pred"]

        items = []
        for year in entry["years"]:
            ds = _load_lstm_ds(
                glacier=rgi_id,
                year=year,
                path_pred_lstm=path_pred,
                period=period,
                apply_smoothing_fn=apply_gaussian_filter
                if apply_smoothing else None,
            )
            if ds is not None and "pred_masked" in ds:
                items.append((year, ds, entry))

        if not items:
            continue

        row_vmin = min(float(ds["pred_masked"].min()) for _, ds, _ in items)
        row_vmax = max(float(ds["pred_masked"].max()) for _, ds, _ in items)
        cmap, norm = get_color_maps(row_vmin, row_vmax)

        # split into sub-rows if more years than ncols
        for i in range(0, len(items), ncols):
            row_data.append({
                "glacier_name": glacier_name,
                "items": items[i:i + ncols],
                "vmin": row_vmin,
                "vmax": row_vmax,
                "cmap": cmap,
                "norm": norm,
            })

    if not row_data:
        raise RuntimeError(
            "No predictions found for any (glacier, year) combination")

    nrows = len(row_data)

    fig = plt.figure(
        figsize=(ncols * figsize_col * cm + 1.2 * cm,
                 nrows * figsize_row * cm),
        dpi=200,
    )
    gs = gridspec.GridSpec(
        nrows,
        ncols + 1,  # extra narrow column for this row's colorbar
        width_ratios=[1] * ncols + [cb_width_ratio],
        wspace=0.45,
        hspace=0.25,
        figure=fig,
    )

    #label_iter = iter(string.ascii_lowercase)
    label_iter = iter(string.ascii_lowercase +
                      string.ascii_uppercase)  # 52 labels instead of 26

    for row_idx, row in enumerate(row_data):
        glacier_name = row["glacier_name"]
        cmap, norm = row["cmap"], row["norm"]
        vmin_display = max(row["vmin"], -12)
        vmax_display = min(row["vmax"], 5)
        mappable_for_cb = None

        for col_idx in range(ncols):
            ax = fig.add_subplot(gs[row_idx, col_idx])

            if col_idx >= len(row["items"]):
                ax.set_axis_off()
                continue

            year, ds, entry = row["items"][col_idx]
            df_stakes = entry.get("df_stakes")

            mappable_for_cb = ds["pred_masked"].plot.imshow(
                ax=ax,
                cmap=cmap,
                norm=norm,
                add_colorbar=False,
            )
            # ax.set_aspect("auto")

            ax.set_title(
                f"{glacier_name.capitalize()} ({year})",
                fontsize=text_size + 1,
            )

            rmse = _stake_overlay_transformer(
                ax=ax,
                glacier=glacier_name,
                year=year,
                cmap=cmap,
                norm=norm,
                ds_pred=ds,
                df_stakes=df_stakes,
                period=period,
            )

            mean_m = float(ds["pred_masked"].mean())
            annotation = (f"RMSE: {rmse:.2f}\n"
                          if rmse else "") + f"mean MB: {mean_m:.2f}"

            if glacier_name.lower() == "rhone":
                x_txt, y_txt = 0.96, 0.04
                ha_txt = "right"
            else:
                x_txt, y_txt = 0.04, 0.03
                ha_txt = "left"

            ax.text(x_txt,
                    y_txt,
                    annotation,
                    transform=ax.transAxes,
                    fontsize=text_size,
                    ha=ha_txt,
                    va="bottom")
            ax.text(0.03,
                    0.97,
                    f"({next(label_iter)})",
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=text_size)
            ax.grid(alpha=0.2)
            ax.tick_params(axis="both", labelsize=text_size - 1)
            ax.set_xlabel("")
            ax.set_ylabel("")

        # ── one colorbar per row ──
        if mappable_for_cb is not None:
            ax_cb = fig.add_subplot(gs[row_idx, ncols])
            cb = fig.colorbar(mappable_for_cb, cax=ax_cb)
            cb.ax.set_ylim(vmin_display, vmax_display)
            if row["vmin"] < vmin_display:
                ax_cb.plot([0.5], [-0.02],
                           marker="v",
                           color="black",
                           markersize=4,
                           transform=ax_cb.transAxes,
                           clip_on=False)
            if row["vmax"] > vmax_display:
                ax_cb.plot([0.5], [1.02],
                           marker="^",
                           color="black",
                           markersize=4,
                           transform=ax_cb.transAxes,
                           clip_on=False)
            cb.set_label("MB [m w.e.]", fontsize=text_size)
            cb.ax.tick_params(labelsize=text_size - 1)

    return fig

In [ ]:
from papers.Transformer_TF.scripts.plotting.pooled_pred import (
    _stake_overlay_transformer,
    aggregate_gridded_transformer_by_glacier,
)


def plot_pixel_scatter_zs_vs_ft(ds_zs, ds_ft, ax, fontsize=8):
    """
    Pixel-wise scatter of multi-region Zero-Shot vs multi-region
    Fine-Tuned predictions. Both come from _load_lstm_ds off the same
    masked WGS84 grid, so no reprojection/interpolation needed.
    """
    if (ds_zs is None or ds_ft is None or "pred_masked" not in ds_zs
            or "pred_masked" not in ds_ft):
        ax.set_axis_off()
        return

    zs = ds_zs["pred_masked"]
    ft = ds_ft["pred_masked"]

    if zs.sizes != ft.sizes:
        ft = ft.interp(lat=zs["lat"], lon=zs["lon"], method="nearest")

    mask = np.isfinite(zs.values) & np.isfinite(ft.values)
    x = zs.values[mask]
    y = ft.values[mask]

    if x.size == 0:
        ax.text(0.5, 0.5, "No overlap", ha="center", va="center")
        ax.set_axis_off()
        return

    ax.scatter(x, y, s=4, alpha=0.3, color='#bababa')

    lims = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lims, lims, "k--", lw=0.8)

    r = np.corrcoef(x, y)[0, 1]
    rmse = root_mean_squared_error(x, y)

    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.3)
    ax.text(0.05,
            0.05,
            f"N = {x.size}\nr = {r:.2f}\nRMSE = {rmse:.2f}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=fontsize)

    ax.set_xlabel("Zero-shot MB", fontsize=fontsize)
    ax.set_ylabel("Fine-tuned MB", fontsize=fontsize)
    ax.yaxis.set_label_coords(-0.28, 0.63)
    ax.tick_params(axis="both", which="both", labelsize=8)
    ax.set_title('')


In [ ]:
import string

# ---- helpers -------------------------------------------------------------


def lookup_glacier_config(glacier_name):
    """
    Case-insensitive lookup into ALL_GLACIER_CONFIG. Returns the *canonical*
    key (the casing that matches df_stakes['GLACIER']) plus its config,
    so stake filtering always uses the right string regardless of how you
    typed the name.
    """
    if glacier_name in ALL_GLACIER_CONFIG:
        return glacier_name, ALL_GLACIER_CONFIG[glacier_name]
    for key in ALL_GLACIER_CONFIG:
        if key.lower() == glacier_name.lower():
            return key, ALL_GLACIER_CONFIG[key]
    raise KeyError(f"{glacier_name!r} not found in ALL_GLACIER_CONFIG")


def zs_ft_paths(gcfg, rep=0):
    rgi_id_str = gcfg["rgi_id_str"]
    ft_local_key = gcfg["ft_local_key"]
    zs_path = os.path.join(PRED_BASE, f"RGI_{rgi_id_str}_ZS")
    ft_path = (os.path.join(PRED_BASE,
                            f"RGI_{rgi_id_str}_FT_{ft_local_key}_rep{rep}")
               if ft_local_key else None)
    return zs_path, ft_path


In [ ]:
# ---- config: each row is a list of up to 2 (glacier, year) cells ---------

YEAR = 2021
MAP_ROWS = [
    [
        {
            "glacier_name": "Argentiere",
            "year": YEAR,
            "text_pos": "right"
        },
        {
            "glacier_name": "aletsch",
            "year": YEAR
        },
    ],
    [
        {
            "glacier_name": "Breidamerkurjoekull",
            "year": YEAR
        },
        {
            "glacier_name": "Nigardsbreen",
            "year": 2020
        },
    ],
    [        {
            "glacier_name": "NORDENSKIOELDBREEN",
            "year": YEAR,
            "text_pos": "right"
        },
        {
            "glacier_name": "LEMON CREEK",
            "year": YEAR,
            "text_pos": "right"
        },
    ],
    [
        {
            "glacier_name": 'TS.TUYUKSUYSKIY',
            "year": YEAR
        },
        {
            "glacier_name": 'Golubin',
            "year": YEAR
        },  # same glacier, two years — still works
    ],
]

GRAD_GLACIERS = [cell["glacier_name"] for row in MAP_ROWS for cell in row]

FT_REP = 0
period = "annual"
cm = 1 / 2.54

ylabel_size = 8
title_size = 9
label_size = 9
text_size = 8
tick_size = 8

n_map_rows = len(MAP_ROWS)
n_grad_cols = len(GRAD_GLACIERS)

# =====================================================
# --- Figure + Grid setup ---
# =====================================================
fig = plt.figure(figsize=(40 * cm, (n_map_rows * 11 + 9) * cm), dpi=300)

gs_global = gridspec.GridSpec(
    2,
    1,
    figure=fig,
    height_ratios=[n_map_rows * (3.2 / 3), 1],
    hspace=0.1,
)

gs_maps = gs_global[0].subgridspec(
    n_map_rows,
    7,
    width_ratios=[1, 1, 0.7, 1, 1, 0.7, 0.05],
    wspace=0.4,
    hspace=0.3,
)

gs_grad = gs_global[1].subgridspec(1,
                                   n_grad_cols,
                                   width_ratios=[1] * n_grad_cols,
                                   wspace=0.35)

#label_iter = iter(string.ascii_lowercase)
label_iter = iter(string.ascii_lowercase +
                  string.ascii_uppercase)  # 52 labels instead of 26
# =====================================================
# --- Map rows: ZS vs FT ---
# =====================================================
for r, row_cells in enumerate(MAP_ROWS):
    # resolve each cell (glacier, year) into its own config/paths — cells
    # in the same row can now belong to completely different regions
    resolved_cells = []
    for cell_cfg in row_cells:
        glacier_name, gcfg = lookup_glacier_config(cell_cfg["glacier_name"])
        ft_rep = cell_cfg.get("ft_rep", FT_REP)
        zs_path, ft_path = zs_ft_paths(gcfg, rep=ft_rep)
        resolved_cells.append({
            "glacier_name": glacier_name,
            "year": cell_cfg["year"],
            "rgi_id": gcfg["rgi_ids"][0],
            "df_stakes": gcfg["df_stakes"],
            "zs_path": zs_path,
            "ft_path": ft_path,
            "text_pos": cell_cfg.get("text_pos", "left"),
        })

    # --- 1) vmin/vmax across every cell in the row, both models ---
    row_vals = []
    for cell in resolved_cells:
        if cell["ft_path"] is None:
            continue
        ds_zs_tmp = _load_lstm_ds(glacier=cell["rgi_id"],
                                  year=cell["year"],
                                  path_pred_lstm=cell["zs_path"],
                                  period=period,
                                  apply_smoothing_fn=apply_gaussian_filter)
        ds_ft_tmp = _load_lstm_ds(glacier=cell["rgi_id"],
                                  year=cell["year"],
                                  path_pred_lstm=cell["ft_path"],
                                  period=period,
                                  apply_smoothing_fn=apply_gaussian_filter)
        for ds_tmp in (ds_zs_tmp, ds_ft_tmp):
            if ds_tmp is not None and "pred_masked" in ds_tmp:
                row_vals += [
                    float(ds_tmp["pred_masked"].min()),
                    float(ds_tmp["pred_masked"].max())
                ]

    if len(row_vals) == 0:
        print(f"  [skip] row {r}: no predictions found")
        continue

    row_vmin = max(min(row_vals), -12)
    row_vmax = max(row_vals)
    vmax_display = min(row_vmax, 4.0)
    vmin_display = max(row_vmin, -12)
    cmap, norm = get_color_maps(row_vmin, row_vmax)

    ax_cb = fig.add_subplot(gs_maps[r, 6])
    last_mappable = None

    # --- 2) plot each cell (up to 2 per row) ---
    for j, cell in enumerate(resolved_cells):
        if cell["ft_path"] is None:
            print(
                f"  [skip] {cell['glacier_name']}: no fine-tuned model for this region"
            )
            continue

        glacier_name = cell["glacier_name"]
        year = cell["year"]
        rgi_id = cell["rgi_id"]
        df_stakes = cell["df_stakes"]
        text_pos = cell["text_pos"]

        if j == 0:
            ax_zs = fig.add_subplot(gs_maps[r, 0])
            ax_ft = fig.add_subplot(gs_maps[r, 1])
            ax_s = fig.add_subplot(gs_maps[r, 2])
        else:
            ax_zs = fig.add_subplot(gs_maps[r, 3])
            ax_ft = fig.add_subplot(gs_maps[r, 4])
            ax_s = fig.add_subplot(gs_maps[r, 5])

        ds_zs = _load_lstm_ds(glacier=rgi_id,
                              year=year,
                              path_pred_lstm=cell["zs_path"],
                              period=period,
                              apply_smoothing_fn=apply_gaussian_filter)
        ds_ft = _load_lstm_ds(glacier=rgi_id,
                              year=year,
                              path_pred_lstm=cell["ft_path"],
                              period=period,
                              apply_smoothing_fn=apply_gaussian_filter)

        img_zs = None
        if ds_zs is not None and "pred_masked" in ds_zs:
            img_zs = ds_zs["pred_masked"].plot.imshow(ax=ax_zs,
                                                      cmap=cmap,
                                                      norm=norm,
                                                      add_colorbar=False)
            ax_zs.set_title(f"{glacier_name.capitalize()} ({year})\nZero-shot",
                            fontsize=title_size)

            mean_zs = float(ds_zs["pred_masked"].mean())
            var_zs = float(ds_zs["pred_masked"].var())
            rmse_zs = _stake_overlay_transformer(
                ax=ax_zs,
                glacier=glacier_name,
                year=year,
                cmap=cmap,
                norm=norm,
                ds_pred=ds_zs,
                df_stakes=df_stakes,
                period=period,
            )
            txt = (f"RMSE: {rmse_zs:.2f}\n" if rmse_zs else
                   "") + f"mean MB: {mean_zs:.2f}\nvar: {var_zs:.2f}"
            x_txt, y_txt, ha_txt = (
                0.96, 0.04, "right") if text_pos == "right" else (0.04, 0.03,
                                                                  "left")
            ax_zs.text(x_txt,
                       y_txt,
                       txt,
                       transform=ax_zs.transAxes,
                       ha=ha_txt,
                       va="bottom",
                       fontsize=text_size)
            ax_zs.grid(alpha=0.2)
            last_mappable = img_zs

        img_ft = None
        if ds_ft is not None and "pred_masked" in ds_ft:
            img_ft = ds_ft["pred_masked"].plot.imshow(ax=ax_ft,
                                                      cmap=cmap,
                                                      norm=norm,
                                                      add_colorbar=False)
            ax_ft.set_title(
                f"{glacier_name.capitalize()} ({year})\nFine-tuned",
                fontsize=title_size)

            mean_ft = float(ds_ft["pred_masked"].mean())
            var_ft = float(ds_ft["pred_masked"].var())
            rmse_ft = _stake_overlay_transformer(
                ax=ax_ft,
                glacier=glacier_name,
                year=year,
                cmap=cmap,
                norm=norm,
                ds_pred=ds_ft,
                df_stakes=df_stakes,
                period=period,
            )
            txt = (f"RMSE: {rmse_ft:.2f}\n" if rmse_ft else
                   "") + f"mean MB: {mean_ft:.2f}\nvar: {var_ft:.2f}"
            x_txt, y_txt, ha_txt = (
                0.96, 0.04, "right") if text_pos == "right" else (0.04, 0.03,
                                                                  "left")
            ax_ft.text(x_txt,
                       y_txt,
                       txt,
                       transform=ax_ft.transAxes,
                       ha=ha_txt,
                       va="bottom",
                       fontsize=text_size)
            ax_ft.grid(alpha=0.2)
            last_mappable = img_ft

        plot_pixel_scatter_zs_vs_ft(ds_zs=ds_zs,
                                    ds_ft=ds_ft,
                                    ax=ax_s,
                                    fontsize=text_size)
        ax_s.set_title("ZS vs FT", fontsize=title_size)

        ax_zs.set_ylabel("Latitude", fontsize=ylabel_size)
        ax_zs.yaxis.set_label_coords(-0.35 if j == 0 else -0.08, 0.63)
        ax_ft.set_ylabel("")
        ax_zs.set_xlabel("Longitude", fontsize=ylabel_size)
        ax_ft.set_xlabel("Longitude", fontsize=ylabel_size)
        ax_zs.tick_params(axis="x", labelsize=tick_size)
        ax_ft.tick_params(axis="x", labelsize=tick_size)

        if j == 0:
            ax_zs.tick_params(axis="y",
                              left=True,
                              labelleft=True,
                              labelsize=tick_size)
        else:
            ax_zs.tick_params(axis="y", left=False, labelleft=False)
        ax_ft.tick_params(labelleft=False, labelsize=tick_size)

        for ax in (ax_zs, ax_ft, ax_s):
            ax.text(0.04,
                    0.98,
                    f"({next(label_iter)})",
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=label_size)

    # --- one colorbar for the row, regardless of how many cells it had ---
    if last_mappable is not None:
        cb = fig.colorbar(last_mappable, cax=ax_cb)
        cb.ax.set_ylim(vmin_display, vmax_display)
        if min(row_vals) < vmin_display:
            ax_cb.plot([0.5], [-0.02],
                       marker="v",
                       color="black",
                       markersize=4,
                       clip_on=False,
                       transform=ax_cb.transAxes)
        if max(row_vals) > vmax_display:
            ax_cb.plot([0.5], [1.02],
                       marker="^",
                       color="black",
                       markersize=4,
                       clip_on=False,
                       transform=ax_cb.transAxes)
        cb.set_label("Mass Balance [m w.e.]", fontsize=ylabel_size)
        cb.ax.tick_params(labelsize=tick_size)
    else:
        ax_cb.set_axis_off()

# =====================================================
# --- Gradient row: unchanged from before ---
# =====================================================
for c, gl in enumerate(GRAD_GLACIERS):
    ax = fig.add_subplot(gs_grad[0, c])
    glacier_name, gcfg = lookup_glacier_config(gl)
    rgi_id = gcfg["rgi_ids"][0]
    df_stakes_gl = gcfg["df_stakes"]
    zs_path, ft_path = zs_ft_paths(gcfg, rep=FT_REP)

    if ft_path is None:
        ax.set_title(f"{gl} — N/A", fontsize=title_size)
        ax.set_axis_off()
        continue

    df_zs = aggregate_gridded_transformer_by_glacier(rgi_id,
                                                     zs_path,
                                                     period=period)
    df_ft = aggregate_gridded_transformer_by_glacier(rgi_id,
                                                     ft_path,
                                                     period=period)

    for df, label, color in ((df_zs, "Zero-shot", COLOR_ANNUAL),
                             (df_ft, "Fine-tuned", COLOR_WINTER)):
        if df.empty:
            continue
        grouped = df.groupby("altitude_interval")["pred"]
        mean, std = grouped.mean(), grouped.std().fillna(0)
        ax.plot(mean.values, mean.index, color=color, lw=1.2, label=label)
        ax.fill_betweenx(mean.index,
                         mean.values - std.values,
                         mean.values + std.values,
                         alpha=0.15,
                         color=color)

    sub = df_stakes_gl[(df_stakes_gl["GLACIER"] == glacier_name)
                       & (df_stakes_gl["PERIOD"] == period)].copy()
    if not sub.empty and "POINT_ELEVATION" in sub.columns:
        bin_size = 40
        bins = np.arange(
            np.floor(sub["POINT_ELEVATION"].min() / bin_size) * bin_size,
            np.ceil(sub["POINT_ELEVATION"].max() / bin_size) * bin_size +
            bin_size,
            bin_size,
        )
        sub["alt_bin"] = pd.cut(sub["POINT_ELEVATION"], bins=bins, right=False)
        centers = {
            iv: round((iv.left + iv.right) / 2)
            for iv in sub["alt_bin"].cat.categories
        }
        sub["alt_bin"] = sub["alt_bin"].map(centers)
        stake_mean = sub.groupby("alt_bin")["POINT_BALANCE"].mean()
        ax.scatter(stake_mean.values,
                   stake_mean.index,
                   marker="o",
                   s=18,
                   facecolors="none",
                   edgecolors="black",
                   lw=0.8,
                   label="Stakes",
                   zorder=5)

    ax.axvline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax.set_title(glacier_name.capitalize(), fontsize=title_size, pad=2)
    ax.grid(alpha=0.2)
    ax.tick_params(labelsize=tick_size, pad=2)
    ax.text(0.04,
            0.98,
            f"({next(label_iter)})",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=label_size)
    ax.set_ylabel("Elevation (m a.s.l.)" if c == 0 else "",
                  fontsize=ylabel_size)
    ax.set_xlabel("Mass balance (m w.e.)", fontsize=ylabel_size)

# =====================================================
# --- Shared legend ---
# =====================================================
handles = [
    Patch(facecolor=COLOR_ANNUAL, alpha=0.25, label="Zero-shot band"),
    Line2D([0], [0], color=COLOR_ANNUAL, lw=1.2, label="Zero-shot mean"),
    Patch(facecolor=COLOR_WINTER, alpha=0.25, label="Fine-tuned band"),
    Line2D([0], [0], color=COLOR_WINTER, lw=1.2, label="Fine-tuned mean"),
    Line2D([0], [0],
           marker='o',
           linestyle='None',
           markersize=6,
           markerfacecolor='none',
           markeredgecolor="black",
           markeredgewidth=1.2,
           label="Stakes mean"),
]
fig.legend(handles=handles,
           loc='lower center',
           bbox_to_anchor=(0.5, 0.03),
           ncol=3,
           fontsize=10,
           frameon=True)

plt.subplots_adjust(bottom=0.10)
plt.show()

# save figure:
fig.savefig('figures/paperTF/fig7_application.png')

#### Simplified version:

In [ ]:
# ---- config: each row is a list of up to 2 (glacier, year) cells ---------
# Per-glacier year already works here: every cell carries its own "year",
# so cells in the same row (or the same glacier used twice) can use
# different years — see Nigardsbreen (2020) vs the others (YEAR) below.

YEAR = 2021
MAP_ROWS = [
    [
        {
            "glacier_name": "Argentiere",
            "year": YEAR,
            "text_pos": "right"
        },
        {
            "glacier_name": "aletsch",
            "year": YEAR
        },
    ],
    # [
    #     {
    #         "glacier_name": "Breidamerkurjoekull",
    #         "year": YEAR
    #     },
    #     {
    #         "glacier_name": "Nigardsbreen",
    #         "year": 2020
    #     },
    # ],
    [
        {
            "glacier_name": "NORDENSKIOELDBREEN",
            "year": YEAR,
            "text_pos": "right"
        },
        {
            "glacier_name": "LEMON CREEK",
            "year": YEAR,
            "text_pos": "right"
        },
    ],
    [
        {
            "glacier_name": 'TS.TUYUKSUYSKIY',
            "year": YEAR
        },
        {
            "glacier_name": 'Golubin',
            "year": YEAR
        },  # same glacier, two years — still works
    ],
]

FT_REP = 0
period = "annual"
cm = 1 / 2.54

ylabel_size = 8
title_size = 12
label_size = 9
text_size = 8
tick_size = 8

n_map_rows = len(MAP_ROWS)

# =====================================================
# --- Figure + Grid setup (maps only, no gradient row) ---
# =====================================================
fig = plt.figure(figsize=(40 * cm, n_map_rows * 11 * cm), dpi=300)

# 3 columns per row: [glacier pair 1, glacier pair 2, colorbar].
# Each pair gets its own tight 2-column subgrid (zs, ft) below, so the
# zs/ft maps within a pair sit close together while the gap *between*
# pairs (and to the colorbar) is controlled separately.
gs_maps = gridspec.GridSpec(
    n_map_rows,
    3,
    figure=fig,
    width_ratios=[2, 2, 0.05],
    wspace=0.25,
    hspace=0.3,
)

#label_iter = iter(string.ascii_lowercase)
label_iter = iter(string.ascii_lowercase +
                  string.ascii_uppercase)  # 52 labels instead of 26
# =====================================================
# --- Map rows: ZS vs FT ---
# =====================================================
for r, row_cells in enumerate(MAP_ROWS):
    # resolve each cell (glacier, year) into its own config/paths — cells
    # in the same row can now belong to completely different regions
    resolved_cells = []
    for cell_cfg in row_cells:
        glacier_name, gcfg = lookup_glacier_config(cell_cfg["glacier_name"])
        ft_rep = cell_cfg.get("ft_rep", FT_REP)
        zs_path, ft_path = zs_ft_paths(gcfg, rep=ft_rep)
        resolved_cells.append({
            "glacier_name": glacier_name,
            "year": cell_cfg["year"],
            "rgi_id": gcfg["rgi_ids"][0],
            "df_stakes": gcfg["df_stakes"],
            "zs_path": zs_path,
            "ft_path": ft_path,
            "text_pos": cell_cfg.get("text_pos", "left"),
        })

    # --- 1) vmin/vmax across every cell in the row, both models ---
    row_vals = []
    for cell in resolved_cells:
        if cell["ft_path"] is None:
            continue
        ds_zs_tmp = _load_lstm_ds(glacier=cell["rgi_id"],
                                  year=cell["year"],
                                  path_pred_lstm=cell["zs_path"],
                                  period=period,
                                  apply_smoothing_fn=apply_gaussian_filter)
        ds_ft_tmp = _load_lstm_ds(glacier=cell["rgi_id"],
                                  year=cell["year"],
                                  path_pred_lstm=cell["ft_path"],
                                  period=period,
                                  apply_smoothing_fn=apply_gaussian_filter)
        for ds_tmp in (ds_zs_tmp, ds_ft_tmp):
            if ds_tmp is not None and "pred_masked" in ds_tmp:
                row_vals += [
                    float(ds_tmp["pred_masked"].min()),
                    float(ds_tmp["pred_masked"].max())
                ]

    if len(row_vals) == 0:
        print(f"  [skip] row {r}: no predictions found")
        continue

    row_vmin = max(min(row_vals), -12)
    row_vmax = max(row_vals)
    vmax_display = min(row_vmax, 4.0)
    vmin_display = max(row_vmin, -12)
    cmap, norm = get_color_maps(row_vmin, row_vmax)

    ax_cb = fig.add_subplot(gs_maps[r, 2])
    last_mappable = None

    # tight zs/ft subgrid for each glacier pair in this row
    gs_pair0 = gs_maps[r, 0].subgridspec(1, 2, width_ratios=[1, 1], wspace=0.05)
    gs_pair1 = gs_maps[r, 1].subgridspec(1, 2, width_ratios=[1, 1], wspace=0.05)

    # --- 2) plot each cell (up to 2 per row) ---
    for j, cell in enumerate(resolved_cells):
        if cell["ft_path"] is None:
            print(
                f"  [skip] {cell['glacier_name']}: no fine-tuned model for this region"
            )
            continue

        glacier_name = cell["glacier_name"]
        year = cell["year"]
        rgi_id = cell["rgi_id"]
        df_stakes = cell["df_stakes"]
        text_pos = cell["text_pos"]

        gs_pair = gs_pair0 if j == 0 else gs_pair1
        ax_zs = fig.add_subplot(gs_pair[0, 0])
        ax_ft = fig.add_subplot(gs_pair[0, 1])

        ds_zs = _load_lstm_ds(glacier=rgi_id,
                              year=year,
                              path_pred_lstm=cell["zs_path"],
                              period=period,
                              apply_smoothing_fn=apply_gaussian_filter)
        ds_ft = _load_lstm_ds(glacier=rgi_id,
                              year=year,
                              path_pred_lstm=cell["ft_path"],
                              period=period,
                              apply_smoothing_fn=apply_gaussian_filter)

        img_zs = None
        if ds_zs is not None and "pred_masked" in ds_zs:
            img_zs = ds_zs["pred_masked"].plot.imshow(ax=ax_zs,
                                                      cmap=cmap,
                                                      norm=norm,
                                                      add_colorbar=False)
            ax_zs.set_title(f"{glacier_name.capitalize()} ({year})\nZero-shot",
                            fontsize=title_size)

            mean_zs = float(ds_zs["pred_masked"].mean())
            var_zs = float(ds_zs["pred_masked"].var())
            rmse_zs = _stake_overlay_transformer(
                ax=ax_zs,
                glacier=glacier_name,
                year=year,
                cmap=cmap,
                norm=norm,
                ds_pred=ds_zs,
                df_stakes=df_stakes,
                period=period,
            )
            txt = (f"RMSE: {rmse_zs:.2f}\n" if rmse_zs else
                   "") + f"mean MB: {mean_zs:.2f}\nvar: {var_zs:.2f}"
            x_txt, y_txt, ha_txt = (
                0.96, 0.04, "right") if text_pos == "right" else (0.04, 0.03,
                                                                  "left")
            ax_zs.text(x_txt,
                       y_txt,
                       txt,
                       transform=ax_zs.transAxes,
                       ha=ha_txt,
                       va="bottom",
                       fontsize=text_size)
            ax_zs.grid(alpha=0.2)
            last_mappable = img_zs

        img_ft = None
        if ds_ft is not None and "pred_masked" in ds_ft:
            img_ft = ds_ft["pred_masked"].plot.imshow(ax=ax_ft,
                                                      cmap=cmap,
                                                      norm=norm,
                                                      add_colorbar=False)
            ax_ft.set_title(
                f"{glacier_name.capitalize()} ({year})\nFine-tuned",
                fontsize=title_size)

            mean_ft = float(ds_ft["pred_masked"].mean())
            var_ft = float(ds_ft["pred_masked"].var())
            rmse_ft = _stake_overlay_transformer(
                ax=ax_ft,
                glacier=glacier_name,
                year=year,
                cmap=cmap,
                norm=norm,
                ds_pred=ds_ft,
                df_stakes=df_stakes,
                period=period,
            )
            txt = (f"RMSE: {rmse_ft:.2f}\n" if rmse_ft else
                   "") + f"mean MB: {mean_ft:.2f}\nvar: {var_ft:.2f}"
            x_txt, y_txt, ha_txt = (
                0.96, 0.04, "right") if text_pos == "right" else (0.04, 0.03,
                                                                  "left")
            ax_ft.text(x_txt,
                       y_txt,
                       txt,
                       transform=ax_ft.transAxes,
                       ha=ha_txt,
                       va="bottom",
                       fontsize=text_size)
            ax_ft.grid(alpha=0.2)
            last_mappable = img_ft

        ax_zs.set_ylabel("", fontsize=ylabel_size)
        ax_zs.yaxis.set_label_coords(-0.35 if j == 0 else -0.08, 0.63)
        ax_ft.set_ylabel("")
        #ax_zs.set_xlabel("Longitude", fontsize=ylabel_size)
        #ax_ft.set_xlabel("Longitude", fontsize=ylabel_size)
        ax_zs.set_xlabel("", fontsize=ylabel_size)
        ax_ft.set_xlabel("", fontsize=ylabel_size)
        ax_zs.tick_params(axis="x", labelsize=tick_size)
        ax_ft.tick_params(axis="x", labelsize=tick_size)

        #if j == 0:
        ax_zs.tick_params(axis="y",
                            left=True,
                            labelleft=True,
                            labelsize=tick_size)
        # else:
        #     ax_zs.tick_params(axis="y", left=False, labelleft=False)
        ax_ft.tick_params(labelleft=False, labelsize=tick_size)

        # for ax in (ax_zs, ax_ft):
        #     ax.text(0.04,
        #             0.98,
        #             f"({next(label_iter)})",
        #             transform=ax.transAxes,
        #             ha="left",
        #             va="top",
        #             fontsize=label_size)

    # --- one colorbar for the row, regardless of how many cells it had ---
    if last_mappable is not None:
        cb = fig.colorbar(last_mappable, cax=ax_cb)
        cb.ax.set_ylim(vmin_display, vmax_display)
        if min(row_vals) < vmin_display:
            ax_cb.plot([0.5], [-0.02],
                       marker="v",
                       color="black",
                       markersize=4,
                       clip_on=False,
                       transform=ax_cb.transAxes)
        if max(row_vals) > vmax_display:
            ax_cb.plot([0.5], [1.02],
                       marker="^",
                       color="black",
                       markersize=4,
                       clip_on=False,
                       transform=ax_cb.transAxes)
        cb.set_label("Mass Balance [m w.e.]", fontsize=ylabel_size)
        cb.ax.tick_params(labelsize=tick_size)
    else:
        ax_cb.set_axis_off()

plt.show()

# save figure:
fig.savefig('figures/paperTF/fig7_application_simplified.png')

In [ ]:
last_year = (
    dfs["01"].dropna(subset=["POINT_BALANCE"
                             ])  # only count actual measurements
    .groupby("GLACIER")["YEAR"].max().sort_values(ascending=False))
print(last_year)

In [ ]:
last_year = (
    dfs["07"].dropna(subset=["POINT_BALANCE"
                             ])  # only count actual measurements
    .groupby("GLACIER")["YEAR"].max().sort_values(ascending=False))
print(last_year)

In [ ]:
last_year = (
    dfs["11"].dropna(subset=["POINT_BALANCE"
                             ])  # only count actual measurements
    .groupby("GLACIER")["YEAR"].max().sort_values(ascending=False))
print(last_year)

In [ ]:
last_year = (
    dfs["08"].dropna(subset=["POINT_BALANCE"
                             ])  # only count actual measurements
    .groupby("GLACIER")["YEAR"].max().sort_values(ascending=False))
print(last_year)

### Gradients all:

In [ ]:
import itertools


def _panel_label_generator():
    """a, b, ..., z, aa, ab, ..., az, ba, ... — never runs out like a plain 26-letter iterator would."""
    n = 1
    while True:
        for combo in itertools.product(string.ascii_lowercase, repeat=n):
            yield "".join(combo)
        n += 1


def plot_gradients_grid(
    glacier_names,
    ncols=4,
    period="annual",
    ft_rep=0,
    figsize_col=6,
    figsize_row=6,
    cm=1 / 2.54,
    title_size=9,
    label_size=9,
    tick_size=8,
    ylabel_size=8,
    legend_height_cm=2.2,
):
    """
    ZS vs FT elevation-gradient panels (with stake overlay), one glacier
    per cell, wrapping ncols per row. Glaciers with no resolvable config,
    no FT model, or no predictions on disk are dropped entirely before
    laying out the grid — no blank panels for them.
    """
    # ---- resolve + fetch data first, drop anything with nothing to plot ----
    entries = []
    for gl in glacier_names:
        try:
            glacier_name, gcfg = lookup_glacier_config(gl)
        except KeyError as e:
            print(f"  [skip] {gl}: {e}")
            continue

        rgi_id = gcfg["rgi_ids"][0]
        df_stakes_gl = gcfg["df_stakes"]
        zs_path, ft_path = zs_ft_paths(gcfg, rep=ft_rep)

        if ft_path is None:
            print(f"  [skip] {glacier_name}: no fine-tuned model for this region")
            continue

        df_zs = aggregate_gridded_transformer_by_glacier(rgi_id, zs_path, period=period)
        df_ft = aggregate_gridded_transformer_by_glacier(rgi_id, ft_path, period=period)

        if df_zs.empty and df_ft.empty:
            print(f"  [skip] {glacier_name}: no predictions found")
            continue

        entries.append({
            "glacier_name": glacier_name,
            "df_stakes": df_stakes_gl,
            "df_zs": df_zs,
            "df_ft": df_ft,
        })

    if not entries:
        raise RuntimeError("No glaciers with predictions found to plot.")

    # ---- now lay out only what's actually going to be drawn ----
    nrows = math.ceil(len(entries) / ncols)
    fig_w = ncols * figsize_col * cm
    fig_h = nrows * figsize_row * cm + legend_height_cm * cm

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(fig_w, fig_h),
        dpi=300,
        squeeze=False,
    )

    label_iter = _panel_label_generator()

    for idx, entry in enumerate(entries):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]

        glacier_name = entry["glacier_name"]
        df_stakes_gl = entry["df_stakes"]
        df_zs = entry["df_zs"]
        df_ft = entry["df_ft"]

        for df, label, color in ((df_zs, "Zero-shot", COLOR_ANNUAL),
                                 (df_ft, "Fine-tuned", COLOR_WINTER)):
            if df.empty:
                continue
            grouped = df.groupby("altitude_interval")["pred"]
            mean, std = grouped.mean(), grouped.std().fillna(0)
            ax.plot(mean.values, mean.index, color=color, lw=1.2, label=label)
            ax.fill_betweenx(mean.index, mean.values - std.values, mean.values + std.values,
                             alpha=0.15, color=color)

        sub = df_stakes_gl[(df_stakes_gl["GLACIER"] == glacier_name)
                           & (df_stakes_gl["PERIOD"] == period)].copy()
        if not sub.empty and "POINT_ELEVATION" in sub.columns:
            bin_size = 40
            bins = np.arange(
                np.floor(sub["POINT_ELEVATION"].min() / bin_size) * bin_size,
                np.ceil(sub["POINT_ELEVATION"].max() / bin_size) * bin_size + bin_size,
                bin_size,
            )
            sub["alt_bin"] = pd.cut(sub["POINT_ELEVATION"], bins=bins, right=False)
            centers = {iv: round((iv.left + iv.right) / 2) for iv in sub["alt_bin"].cat.categories}
            sub["alt_bin"] = sub["alt_bin"].map(centers)
            stake_mean = sub.groupby("alt_bin")["POINT_BALANCE"].mean()
            ax.scatter(stake_mean.values, stake_mean.index, marker="o", s=18,
                      facecolors="none", edgecolors="black", lw=0.8, label="Stakes", zorder=5)

        ax.axvline(0, color="k", lw=0.6, ls="--", alpha=0.5)
        ax.set_title(glacier_name.capitalize(), fontsize=title_size, pad=2)
        ax.grid(alpha=0.2)
        ax.tick_params(labelsize=tick_size, pad=2)
        ax.text(0.04, 0.98, f"({next(label_iter)})", transform=ax.transAxes,
                ha="left", va="top", fontsize=label_size)
        ax.set_ylabel("Elevation (m a.s.l.)" if c == 0 else "", fontsize=ylabel_size)
        ax.set_xlabel("Mass balance (m w.e.)", fontsize=ylabel_size)

    for j in range(len(entries), nrows * ncols):
        axes[j // ncols, j % ncols].set_axis_off()

    bottom_frac = (legend_height_cm * cm) / fig_h
    fig.tight_layout(rect=[0, bottom_frac, 1, 1])

    handles = [
        Patch(facecolor=COLOR_ANNUAL, alpha=0.25, label="Zero-shot band"),
        Line2D([0], [0], color=COLOR_ANNUAL, lw=1.2, label="Zero-shot mean"),
        Patch(facecolor=COLOR_WINTER, alpha=0.25, label="Fine-tuned band"),
        Line2D([0], [0], color=COLOR_WINTER, lw=1.2, label="Fine-tuned mean"),
        Line2D([0], [0], marker='o', linestyle='None', markersize=6,
               markerfacecolor='none', markeredgecolor="black", markeredgewidth=1.2,
               label="Stakes mean"),
    ]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.0),
              ncol=3, fontsize=10, frameon=True)

    return fig

In [ ]:
fig = plot_gradients_grid(list(ALL_REGION_CONFIGS["ALA"].keys()), ncols=5)
plt.show()
fig.savefig('figures/paperTF/appendix/A16_gradients_ALA.png')

In [ ]:
fig = plot_gradients_grid(list(ALL_REGION_CONFIGS["CH"].keys()), ncols=6)
plt.show()
fig.savefig('figures/paperTF/appendix/A11_gradients_CH.png')

In [ ]:
glaciers = list(ALL_REGION_CONFIGS["FR"].keys()) + list(ALL_REGION_CONFIGS["IT"].keys())+ list(ALL_REGION_CONFIGS["AT"].keys())
fig = plot_gradients_grid(glaciers, ncols=6)
plt.show()
fig.savefig('figures/paperTF/appendix/A12_gradients_CEU.png')

In [ ]:
glaciers = list(ALL_REGION_CONFIGS["CA_3"].keys()) + list(ALL_REGION_CONFIGS["CA_4"].keys())
fig = plot_gradients_grid(glaciers, ncols=5)
plt.show()
fig.savefig('figures/paperTF/appendix/A15_gradients_CA.png')

In [ ]:
fig = plot_gradients_grid(list(ALL_REGION_CONFIGS["ISL"].keys()), ncols=8)
plt.show()
fig.savefig('figures/paperTF/appendix/A14_gradients_ISL.png')

In [ ]:
glaciers = list(ALL_REGION_CONFIGS["NOR"].keys()) + list(ALL_REGION_CONFIGS["SJM"].keys())
fig = plot_gradients_grid(glaciers, ncols=6)
plt.show()
fig.savefig('figures/paperTF/appendix/A13_gradients_NOR_SJM.png')

## Geodetic comparison:

In [ ]:
from papers.Transformer_TF.PMB_sources.RGI_11_Switzerland.scripts.geodetic import *
# load geodetic MB for Switzerland
geodetic_mb = get_geodetic_MB(cfg)
periods_per_glacier, geoMB_per_glacier = build_periods_per_glacier(geodetic_mb)
print(f"Glaciers with geodetic periods: {list(periods_per_glacier.keys())}")

In [ ]:
# map glacier short names to RGI IDs (needed to locate zarr files)
ch_name_to_rgi = {
    glacier_name: gcfg["rgi_ids"][0]
    for glacier_name, gcfg in GLACIER_CONFIG.items()
    if gcfg["rgi_id_str"] == "11"
}

# extend to all CH glaciers with geodetic periods
# (not just the ones in GLACIER_CONFIG)
for glacier_name in periods_per_glacier:
    if glacier_name not in ch_name_to_rgi:
        matches = monthly_cache["CH"]["data_monthly"].query(
            f"GLACIER == '{glacier_name}'")["RGIId"].unique()
        if len(matches):
            ch_name_to_rgi[glacier_name] = matches[0]

print(f"Mapped {len(ch_name_to_rgi)} CH glaciers to RGI IDs")

In [ ]:
# ── All CH glaciers with stakes and geodetic periods ──────────────────────────
ch_stake_glaciers_with_geodetic = [
    g for g in periods_per_glacier if g in ch_name_to_rgi
]
print(f"CH glaciers with geodetic periods: {ch_stake_glaciers_with_geodetic}")

# get all their RGI IDs
ch_all_rgi_ids = [ch_name_to_rgi[g] for g in ch_stake_glaciers_with_geodetic]
print(f"RGI IDs: {ch_all_rgi_ids}")

In [ ]:
def process_geodetic_mb_comparison_transformer(
    glacier_list,
    path_SMB_GLAMOS_csv,
    periods_per_glacier,
    geoMB_per_glacier,
    gl_area,
    test_glaciers,
    pred_base,  # e.g. PRED_BASE
    pred_folder,  # e.g. "RGI_11_ZS"
    name_to_rgi,  # dict: glacier_name -> rgi_id
    cfg,
):
    """
    Like process_geodetic_mb_comparison but reads transformer zarr predictions
    stored under RGI IDs rather than glacier names.
    """
    mbm_mb_mean, glamos_mb_mean = [], []
    mbm_mb_var, glamos_mb_var = [], []
    geodetic_mb, geodetic_sigma = [], []
    gl, gl_type, area = [], [], []
    period_len, start_year, end_year = [], [], []

    pred_path = os.path.join(pred_base, pred_folder)

    for glacier_name in tqdm(glacier_list, desc="Processing glaciers"):
        rgi_id = name_to_rgi.get(glacier_name)
        if rgi_id is None:
            print(f"  SKIP {glacier_name}: no RGI ID mapping found")
            continue

        folder_path = os.path.join(pred_path, rgi_id)
        if not os.path.isdir(folder_path):
            print(f"  SKIP {glacier_name}: no predictions at {folder_path}")
            continue

        glamos_file = os.path.join(path_SMB_GLAMOS_csv, "fix",
                                   f"{glacier_name}_fix.csv")
        GLAMOS_glwmb = get_GLAMOS_glwmb(
            glacier_name,
            cfg) if os.path.exists(glamos_file) else pd.DataFrame()

        periods = periods_per_glacier.get(glacier_name, [])
        geo_tuples = geoMB_per_glacier.get(glacier_name, [])
        if not periods or not geo_tuples:
            continue

        for i, period in enumerate(periods):
            if i >= len(geo_tuples) or len(geo_tuples[i]) < 2:
                continue
            if period[1] == 2021 and glacier_name == "silvretta":
                continue

            # check years available under RGI ID naming
            available_years = {
                int(m.group(1))
                for f in os.listdir(folder_path) if (m := re.match(
                    rf"{re.escape(rgi_id)}_(\d{{4}})_annual\.zarr", f))
            }
            expected_years = set(range(period[0], period[1] + 1))
            missing = expected_years - available_years
            if missing:
                print(
                    f"  SKIP {glacier_name} {period}: missing years {sorted(missing)}"
                )
                continue

            mbm_mb, glamos_mb = [], []
            for year in range(period[0], period[1] + 1):
                zarr_path = os.path.join(folder_path,
                                         f"{rgi_id}_{year}_annual.zarr")
                if os.path.exists(zarr_path):
                    ds = xr.open_zarr(zarr_path)
                    mbm_mb.append(float(ds["pred_masked"].mean().values))
                else:
                    mbm_mb.append(np.nan)
                glamos_mb.append(GLAMOS_glwmb["GLAMOS Balance"].get(
                    year, np.nan) if not GLAMOS_glwmb.empty else np.nan)

            mbm_mb_mean.append(np.nanmean(mbm_mb))
            mbm_mb_var.append(np.nanstd(mbm_mb))
            glamos_mb_mean.append(np.nanmean(glamos_mb))
            glamos_mb_var.append(np.nanstd(glamos_mb))
            geodetic_mb.append(geo_tuples[i][0])
            geodetic_sigma.append(geo_tuples[i][1])
            gl.append(glacier_name)
            gl_type.append(glacier_name in test_glaciers)
            period_len.append(period[1] - period[0])
            area.append(gl_area.get(glacier_name, np.nan))
            start_year.append(period[0])
            end_year.append(period[1])

    df_all = pd.DataFrame({
        "MBM MB": mbm_mb_mean,
        "GLAMOS MB": glamos_mb_mean,
        "MBM MB std": mbm_mb_var,
        "GLAMOS MB std": glamos_mb_var,
        "Geodetic MB": geodetic_mb,
        "Geodetic MB sigma": geodetic_sigma,
        "GLACIER": gl,
        "Period Length": period_len,
        "Test Glacier": gl_type,
        "Area": area,
        "start_year": start_year,
        "end_year": end_year,
    }).sort_values("Area")

    return df_all

In [ ]:
PRED_BASE = os.path.join(cfg.dataPath, "predictions_transformer")

# Areas (with clariden alias fix)
gl_area = get_gl_area(cfg)
gl_area["clariden"] = gl_area["claridenL"]

# glaciers to compare — intersection of geodetic periods and available predictions
glacier_list_ch = [g for g in periods_per_glacier if g in ch_name_to_rgi]
print(f"Glaciers to compare: {len(glacier_list_ch)}")

df_geodetic_zs = process_geodetic_mb_comparison_transformer(
    glacier_list=glacier_list_ch,
    path_SMB_GLAMOS_csv=os.path.join(cfg.dataPath, path_SMB_GLAMOS_csv),
    periods_per_glacier=periods_per_glacier,
    geoMB_per_glacier=geoMB_per_glacier,
    gl_area=gl_area,
    test_glaciers=[],  # adjust if you have a test set definition
    pred_base=PRED_BASE,
    pred_folder="RGI_11_ZS",
    name_to_rgi=ch_name_to_rgi,
    cfg=cfg,
)

print(f"Result: {len(df_geodetic_zs)} glacier-period pairs")
df_geodetic_zs.head()

In [ ]:
df_geodetic_zs.GLACIER.unique()

In [ ]:
# Compute RMSE and Pearson correlation
rmse_nn = root_mean_squared_error(df_geodetic_zs["Geodetic MB"],
                                  df_geodetic_zs["MBM MB"])
corr_nn = np.corrcoef(df_geodetic_zs["Geodetic MB"],
                      df_geodetic_zs["MBM MB"])[0, 1]
gl_list = [
    'Schwarzbach', 'Sexrouge', 'Murtel', 'Basodino', 'Adler', 'Hohlaub',
    'Tsanfleuron', 'Silvretta', 'Gries', 'Clariden', 'Gietro', 'Schwarzberg',
    'Allalin', 'Findelen', 'Rhone', 'Corbassiere', 'Aletsch'
]
df_geodetic_zs = df_geodetic_zs[df_geodetic_zs.GLACIER.apply(
    lambda x: x.capitalize() in gl_list)]
fig = plot_mbm_vs_geodetic_by_area_bin(
    df_geodetic_zs,
    bins=[0, 1, 5, 10, 100, np.inf],
    labels=['<1', '1-5', '5–10', '>10', '>100'],
    max_bins=4,
    figsize=(35, 10),
)